In [4]:
#!/usr/bin/env python3
"""
Agente de Conciliación Bancaria — MI EMPRESA SA DE CV vs BBVA México
Concilia movimientos_bancarios.txt vs extracto_bancario_MT940.txt
usando la API de Claude con tool use (loop agéntico).

Llaves de conciliación:
  - Referencia (llave principal)
  - Fecha de operación
  - Monto de la transacción
  - Dirección: Crédito (CR) / Débito (DB)
  - CLABE (validación adicional cuando esté disponible)
"""

import anthropic
import json
import re
from datetime import datetime
from pathlib import Path

CLIENT = anthropic.Anthropic()
MODEL = "claude-opus-4-7"

BASE_DIR = Path(__file__).parent
MOVIMIENTOS_FILE = BASE_DIR / "movimientos_bancarios.txt"
MT940_FILE = BASE_DIR / "extracto_bancario_MT940.txt"
REPORTE_FILE = BASE_DIR / "reporte_conciliacion.txt"

MESES = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12,
}


# ============================================================
# IMPLEMENTACIONES DE HERRAMIENTAS
# ============================================================

def _parsear_movimientos(file_path: str) -> list:
    """Parsea movimientos_bancarios.txt y retorna registros estructurados."""
    content = Path(file_path).read_text(encoding="utf-8")
    records = []

    # Separar bloques por líneas de guiones (80 chars)
    blocks = re.split(r"\n-{60,}\n", content)

    for block in blocks:
        if "Fecha" not in block or "Monto" not in block:
            continue

        record = {}

        # Fecha
        m = re.search(r"Fecha\s*:\s*(\d{2})/(\w+)/(\d{4})", block)
        if m:
            day, month_str, year = int(m.group(1)), m.group(2), int(m.group(3))
            month = MESES.get(month_str, 0)
            if month:
                record["fecha"] = f"{year:04d}-{month:02d}-{day:02d}"
                record["fecha_display"] = f"{day:02d}/{month_str}/{year}"

        # Tipo y dirección
        m = re.search(r"Tipo\s*:\s*(.+)", block)
        if m:
            tipo = m.group(1).strip()
            record["tipo"] = tipo
            record["tipo_dir"] = "CR" if "Crédito" in tipo else "DB"

        # Monto: priorizar "Monto Total" (pagos combinados) sobre "Monto" simple.
        # "Monto Parcial" y "Monto Total" tienen palabras entre "Monto" y ":"
        # por lo que NO son capturados por Monto\s*:
        m = re.search(r"Monto\s+Total\s*:\s*([+-]?\$[\d,]+\.?\d*)", block)
        if not m:
            m = re.search(r"Monto\s*:\s*([+-]?\$[\d,]+\.?\d*)", block)
        if m:
            monto_str = m.group(1).replace("$", "").replace(",", "")
            record["monto"] = float(monto_str)
            record["monto_abs"] = abs(float(monto_str))

        # Referencia (puede ser múltiple: "REF1 / REF2")
        m = re.search(r"Referencia\s*:\s*(.+)", block)
        if m:
            ref_raw = m.group(1).strip()
            record["referencia_raw"] = ref_raw
            record["referencias"] = [r.strip() for r in re.split(r"\s*/\s*", ref_raw)]

        # Concepto
        m = re.search(r"Concepto\s*:\s*(.+)", block)
        if m:
            record["concepto"] = m.group(1).strip()

        # Contraparte (Ordenante para créditos, Beneficiario externo para débitos)
        if record.get("tipo_dir") == "CR":
            m = re.search(r"Ordenante\s*:\s*(.+)", block)
            if m:
                record["contraparte"] = m.group(1).strip()
        else:
            # Tomar primer Beneficiario que no sea la empresa propia
            for candidate in re.findall(r"Beneficiario\s*:\s*(.+)", block):
                if "MI EMPRESA" not in candidate:
                    record["contraparte"] = candidate.strip()
                    break

        # CLABE
        m = re.search(r"CLABE\s+(?:Origen|Destino)\s*:\s*(\d+)", block)
        if m:
            record["clabe"] = m.group(1).strip()

        # RFC contraparte
        m = re.search(r"RFC\s+(?:Ordenante|Beneficiario|Contribuyente)\s*:\s*(\w+)", block)
        if m:
            record["rfc"] = m.group(1).strip()

        # Solo agregar si tiene los campos mínimos
        if record.get("referencias") and record.get("fecha") and "monto_abs" in record:
            records.append(record)

    return records


def _parsear_mt940(file_path: str) -> list:
    """Parsea archivo SWIFT MT940 y retorna registros estructurados.

    Formato :61: (no estándar usado en este archivo):
    YYMMDD[MMDD]<CD|DB><amount,cents><NXXX><descripcion>//<referencia>
    donde CD = Crédito y DB = Débito.
    """
    content = Path(file_path).read_text(encoding="utf-8")
    lines = [ln.rstrip() for ln in content.split("\n")]

    # Agrupar líneas en bloques iniciando en cada :61:
    blocks: list[list[str]] = []
    current: list[str] = []
    for line in lines:
        if line.startswith(":61:") and current:
            blocks.append(current)
            current = [line]
        else:
            current.append(line)
    if current:
        blocks.append(current)

    records = []
    for block in blocks:
        tag61 = next((ln for ln in block if ln.startswith(":61:")), None)
        if not tag61:
            continue

        tag86 = next((ln for ln in block if ln.startswith(":86:")), None)
        record: dict = {}

        rest = tag61[4:]  # quitar ":61:"

        # Fecha valor YYMMDD
        vdate = rest[:6]
        try:
            year = 2000 + int(vdate[:2])
            month = int(vdate[2:4])
            day = int(vdate[4:6])
            record["fecha"] = f"{year:04d}-{month:02d}-{day:02d}"
            rest = rest[6:]
        except ValueError:
            continue

        # Fecha de entrada opcional MMDD (detectar 4 dígitos antes de CD/DB)
        if re.match(r"^\d{4}(?:CD|DB)", rest):
            rest = rest[4:]

        # Indicador CR/DB (no estándar: CD=crédito, DB=débito)
        if rest.startswith("CD"):
            record["tipo_dir"] = "CR"
            rest = rest[2:]
        elif rest.startswith("DB"):
            record["tipo_dir"] = "DB"
            rest = rest[2:]
        elif rest.startswith("C"):
            record["tipo_dir"] = "CR"
            rest = rest[1:]
        elif rest.startswith("D"):
            record["tipo_dir"] = "DB"
            rest = rest[1:]
        else:
            continue

        # Monto (formato: dígitos,centavos — ej. 45000,00)
        am = re.match(r"^(\d+,\d{2})", rest)
        if not am:
            continue
        monto_str = am.group(1).replace(",", ".")
        record["monto_abs"] = float(monto_str)
        record["monto"] = record["monto_abs"] if record["tipo_dir"] == "CR" else -record["monto_abs"]
        rest = rest[len(am.group(0)):]

        # Código de tipo de transacción: N + 3 chars (ej. NTRF, NCHQ, NCHG)
        tm = re.match(r"^(N\w{3})", rest)
        if tm:
            record["tipo_transaccion"] = tm.group(1)
            rest = rest[4:]

        # Descripción // Referencia del cliente
        if "//" in rest:
            partes = rest.split("//", 1)
            record["descripcion"] = partes[0].strip()
            record["referencia"] = partes[1].strip()
        else:
            record["descripcion"] = rest.strip()
            continue  # sin referencia no se puede conciliar

        # Parsear etiqueta :86: — subcampos separados por ?NN
        if tag86:
            subfields: dict[str, str] = {}
            partes86 = re.split(r"\?(\d{2})", tag86[4:])
            k = 1
            while k + 1 < len(partes86):
                subfields[partes86[k]] = partes86[k + 1]
                k += 2
            record["subfields"] = subfields
            if "00" in subfields:
                record["concepto"] = subfields["00"]
            if "21" in subfields:
                record["contraparte"] = subfields["21"]

        records.append(record)

    return records


def _conciliar(movimientos: list, mt940_records: list) -> dict:
    """Concilia movimientos vs MT940.

    Algoritmo:
    1. Para cada movimiento, busca en el pool MT940 todos sus registros
       por referencia + fecha + dirección.
    2. Si encuentra coincidencias, verifica que los montos cuadren.
    3. Clasifica como 1:1 (una referencia) o 1:N (múltiples referencias).
    """
    mt940_pool = dict(enumerate(mt940_records))
    mov_pool = dict(enumerate(movimientos))
    used_mt940: set[int] = set()
    used_mov: set[int] = set()
    reconciled = []

    for i, mov in mov_pool.items():
        refs = mov.get("referencias", [])
        fecha = mov.get("fecha")
        monto_abs = mov.get("monto_abs", 0.0)
        tipo_dir = mov.get("tipo_dir")

        matched_indices: list[int] = []
        total_mt940 = 0.0

        for ref in refs:
            for j, mt940 in mt940_pool.items():
                if j in used_mt940:
                    continue
                if (mt940.get("referencia") == ref
                        and mt940.get("fecha") == fecha
                        and mt940.get("tipo_dir") == tipo_dir):
                    matched_indices.append(j)
                    total_mt940 += mt940.get("monto_abs", 0.0)
                    break  # una sola coincidencia por referencia

        if not matched_indices:
            continue

        diferencia = round(total_mt940 - monto_abs, 2)
        tipo_conc = "1:1" if len(matched_indices) == 1 else f"1:{len(matched_indices)}"
        estado = "CONCILIADO" if abs(diferencia) < 0.01 else "DIFERENCIA_MONTO"

        reconciled.append({
            "tipo_conciliacion": tipo_conc,
            "estado": estado,
            "movimiento": mov,
            "mt940_records": [mt940_pool[k] for k in matched_indices],
            "monto_movimientos": round(monto_abs, 2),
            "monto_mt940": round(total_mt940, 2),
            "diferencia": diferencia,
        })
        used_mt940.update(matched_indices)
        used_mov.add(i)

    no_conc_mov = [mov_pool[i] for i in mov_pool if i not in used_mov]
    no_conc_mt940 = [mt940_pool[j] for j in mt940_pool if j not in used_mt940]

    return {
        "conciliados": reconciled,
        "no_conciliados_movimientos": no_conc_mov,
        "no_conciliados_mt940": no_conc_mt940,
        "resumen": {
            "total_movimientos": len(movimientos),
            "total_mt940": len(mt940_records),
            "conciliados_1_1": sum(1 for r in reconciled if r["tipo_conciliacion"] == "1:1"),
            "conciliados_1_n": sum(1 for r in reconciled if r["tipo_conciliacion"] != "1:1"),
            "no_conciliados_movimientos": len(no_conc_mov),
            "no_conciliados_mt940": len(no_conc_mt940),
            "tasa_conciliacion": round(len(used_mov) / len(movimientos) * 100, 1) if movimientos else 0,
        },
    }


def _generar_reporte(resultado: dict, output_path: str) -> str:
    """Genera un reporte de conciliación detallado y lo guarda en disco."""
    ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    SEP = "=" * 80
    SUB = "-" * 80
    lines = []

    lines += [
        SEP,
        "         REPORTE DE CONCILIACIÓN BANCARIA",
        f"         Generado : {ahora}",
        SEP,
        f"  Empresa  : MI EMPRESA SA DE CV  (RFC: MEM960501AB2)",
        f"  Banco    : BBVA México SA — Cta. 0156 7801234567",
        f"  Periodo  : 01 al 09 de Mayo 2024",
        f"  Fuente 1 : {MOVIMIENTOS_FILE.name}",
        f"  Fuente 2 : {MT940_FILE.name}",
        SEP,
    ]

    res = resultado["resumen"]
    lines += [
        "",
        "RESUMEN EJECUTIVO",
        SUB,
        f"  Registros en Movimientos Bancarios : {res['total_movimientos']:>3}",
        f"  Registros en Extracto MT940        : {res['total_mt940']:>3}",
        f"  ──────────────────────────────────────",
        f"  Conciliados 1:1                    : {res['conciliados_1_1']:>3}",
        f"  Conciliados 1:N (consolidados)     : {res['conciliados_1_n']:>3}",
        f"  No conciliados (Movimientos)       : {res['no_conciliados_movimientos']:>3}",
        f"  No conciliados (MT940)             : {res['no_conciliados_mt940']:>3}",
        f"  ──────────────────────────────────────",
        f"  Tasa de conciliación               : {res['tasa_conciliacion']:>5}%",
        "",
    ]

    lines += [
        SEP,
        "DETALLE DE REGISTROS CONCILIADOS",
        SEP,
    ]

    for idx, item in enumerate(resultado["conciliados"], 1):
        mov = item["movimiento"]
        mt940s = item["mt940_records"]
        icono = "✓" if item["estado"] == "CONCILIADO" else "⚠"

        tipo_mv = mov.get("tipo_dir", "?")
        dir_label = "CRÉDITO" if tipo_mv == "CR" else "DÉBITO "

        lines += [
            "",
            f"  [{idx:02d}] {icono} {item['estado']:<20}  Tipo conciliación: {item['tipo_conciliacion']}",
            f"       Referencia   : {mov.get('referencia_raw', 'N/A')}",
            f"       Fecha        : {mov.get('fecha_display', mov.get('fecha', 'N/A'))}",
            f"       Dirección    : {dir_label}",
            f"       Concepto     : {mov.get('concepto', 'N/A')}",
            f"       Contraparte  : {mov.get('contraparte', 'N/A')}",
        ]
        if mov.get("clabe"):
            lines.append(f"       CLABE        : {mov['clabe']}")
        if mov.get("rfc"):
            lines.append(f"       RFC          : {mov['rfc']}")

        lines += [
            f"       ─────────────────────────────────────────────────────",
            f"       Monto Movim. : ${mov.get('monto_abs', 0):>14,.2f}",
            f"       Monto MT940  : ${item['monto_mt940']:>14,.2f}",
        ]

        if len(mt940s) > 1:
            lines.append(f"       Desglose MT940 ({len(mt940s)} registros combinados):")
            for k, mt940 in enumerate(mt940s, 1):
                concepto_mt940 = mt940.get("concepto", mt940.get("descripcion", "N/A"))
                contraparte_mt940 = mt940.get("contraparte", "")
                desc_extra = f" | {contraparte_mt940}" if contraparte_mt940 else ""
                lines.append(
                    f"         {k}. REF: {mt940.get('referencia', 'N/A'):<18}"
                    f" ${mt940.get('monto_abs', 0):>10,.2f}"
                    f" | {concepto_mt940}{desc_extra}"
                )

        if abs(item.get("diferencia", 0)) >= 0.01:
            lines.append(f"       *** DIFERENCIA DETECTADA: ${abs(item['diferencia']):,.2f} ***")

        lines.append("  " + "─" * 70)

    # Registros no conciliados
    if resultado["no_conciliados_movimientos"]:
        lines += [
            "",
            SEP,
            "REGISTROS SIN CONCILIAR — MOVIMIENTOS BANCARIOS",
            SUB,
            "  (Presentes en el sistema interno pero no encontrados en el extracto bancario)",
            "",
        ]
        for mov in resultado["no_conciliados_movimientos"]:
            lines.append(
                f"  ⚠ REF: {mov.get('referencia_raw', 'N/A'):<25}"
                f" | {mov.get('fecha', 'N/A')}"
                f" | {'CR' if mov.get('tipo_dir') == 'CR' else 'DB'}"
                f" | ${mov.get('monto_abs', 0):>12,.2f}"
                f" | {mov.get('concepto', 'N/A')}"
            )
    else:
        lines += [
            "",
            "  ✓ Todos los registros de Movimientos Bancarios fueron conciliados.",
        ]

    if resultado["no_conciliados_mt940"]:
        lines += [
            "",
            SEP,
            "REGISTROS SIN CONCILIAR — EXTRACTO MT940",
            SUB,
            "  (Presentes en el extracto bancario pero no en el sistema interno)",
            "",
        ]
        for mt940 in resultado["no_conciliados_mt940"]:
            lines.append(
                f"  ⚠ REF: {mt940.get('referencia', 'N/A'):<25}"
                f" | {mt940.get('fecha', 'N/A')}"
                f" | {'CR' if mt940.get('tipo_dir') == 'CR' else 'DB'}"
                f" | ${mt940.get('monto_abs', 0):>12,.2f}"
                f" | {mt940.get('concepto', mt940.get('descripcion', 'N/A'))}"
            )
    else:
        lines += [
            "",
            "  ✓ Todos los registros del Extracto MT940 fueron conciliados.",
        ]

    lines += [
        "",
        SEP,
        "  Fin del Reporte — Generado automáticamente por Agente de Conciliación Bancaria",
        SEP,
    ]

    reporte_texto = "\n".join(lines)
    Path(output_path).write_text(reporte_texto, encoding="utf-8")
    return reporte_texto


# ============================================================
# DEFINICIÓN DE HERRAMIENTAS PARA CLAUDE
# ============================================================

TOOLS = [
    {
        "name": "parsear_movimientos_bancarios",
        "description": (
            "Parsea el archivo movimientos_bancarios.txt y retorna lista de registros "
            "estructurados. Cada registro incluye: fecha (YYYY-MM-DD), tipo (DEPÓSITO/PAGO/CARGO), "
            "tipo_dir (CR=crédito / DB=débito), referencias (lista — puede ser más de una en "
            "pagos combinados), monto_abs, concepto, contraparte, clabe y rfc."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Ruta absoluta al archivo movimientos_bancarios.txt",
                }
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "parsear_mt940",
        "description": (
            "Parsea el archivo en formato SWIFT MT940 y retorna lista de registros "
            "estructurados. Cada registro incluye: fecha (YYYY-MM-DD), tipo_dir (CR/DB), "
            "monto_abs, tipo_transaccion (NTRF=transferencia, NCHQ=cheque, NCHG=cargo bancario), "
            "descripcion, referencia (ej. REF240502001) y concepto del subcampo :86:."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Ruta absoluta al archivo extracto_bancario_MT940.txt",
                }
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "conciliar_registros",
        "description": (
            "Concilia los registros de movimientos bancarios contra los del extracto MT940. "
            "Llaves de conciliación: referencia, fecha, monto y dirección (CR/DB). "
            "Detecta coincidencias 1:1 y consolidaciones 1:N (un movimiento con múltiples "
            "referencias = múltiples registros MT940, como el caso del pago combinado "
            "de renta + CFE con referencias REF240508002/REF240508003). "
            "Retorna: registros conciliados con tipo y estado, registros no conciliados "
            "en cada fuente, y resumen con totales y tasa de conciliación."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "movimientos": {
                    "type": "array",
                    "description": "Lista de registros parseados de movimientos_bancarios.txt",
                },
                "mt940_records": {
                    "type": "array",
                    "description": "Lista de registros parseados del archivo MT940",
                },
            },
            "required": ["movimientos", "mt940_records"],
        },
    },
    {
        "name": "generar_reporte",
        "description": (
            "Genera un reporte de conciliación bancaria en formato legible y lo guarda en disco. "
            "Incluye: encabezado con datos de la empresa y banco, resumen ejecutivo con totales "
            "y tasa de conciliación, detalle de cada registro conciliado (tipo 1:1 o 1:N, "
            "montos comparados, desglose de registros en casos 1:N, diferencias si existen), "
            "y listado de registros no conciliados de cada fuente."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "resultado": {
                    "type": "object",
                    "description": "Resultado de la conciliación retornado por conciliar_registros",
                },
                "output_path": {
                    "type": "string",
                    "description": "Ruta absoluta donde guardar el reporte (.txt)",
                },
            },
            "required": ["resultado", "output_path"],
        },
    },
]


# ============================================================
# DESPACHADOR DE HERRAMIENTAS
# ============================================================

def ejecutar_herramienta(nombre: str, entrada: dict):
    """Ejecuta la herramienta solicitada y retorna el resultado."""
    dispatch = {
        "parsear_movimientos_bancarios": lambda: _parsear_movimientos(entrada["file_path"]),
        "parsear_mt940": lambda: _parsear_mt940(entrada["file_path"]),
        "conciliar_registros": lambda: _conciliar(entrada["movimientos"], entrada["mt940_records"]),
        "generar_reporte": lambda: _generar_reporte(entrada["resultado"], entrada["output_path"]),
    }
    if nombre not in dispatch:
        raise ValueError(f"Herramienta desconocida: {nombre}")
    return dispatch[nombre]()


# ============================================================
# AGENTE PRINCIPAL — LOOP AGÉNTICO
# ============================================================

SYSTEM_PROMPT = """Eres un agente especialista en conciliación bancaria para México.

Tu tarea es conciliar dos archivos del periodo Mayo 2024:
1. movimientos_bancarios.txt — Extracto interno de MI EMPRESA SA DE CV
2. extracto_bancario_MT940.txt — Extracto bancario BBVA México en formato SWIFT MT940

Llaves de conciliación (en orden de prioridad):
1. Referencia (campo principal — ej. REF240502001)
2. Fecha de operación (YYYY-MM-DD)
3. Monto de la transacción (comparar valores absolutos)
4. Dirección: Crédito/CR o Débito/DB
5. CLABE (validación adicional cuando esté disponible)

CASO ESPECIAL — Conciliación 1:N (consolidación):
Un registro en movimientos_bancarios.txt puede contener múltiples referencias separadas
por " / " (ej. "REF240508002 / REF240508003"). Esto indica que un solo movimiento
consolidado en el sistema interno corresponde a 2 o más registros separados en el
extracto MT940. En estos casos el monto total del movimiento debe igualar la suma
de los registros MT940 correspondientes.

Proceso a seguir:
1. Parsear movimientos_bancarios.txt
2. Parsear extracto_bancario_MT940.txt
3. Conciliar ambos archivos identificando coincidencias 1:1 y 1:N
4. Generar el reporte en disco
5. Proporcionar análisis ejecutivo con hallazgos y conclusiones"""


def ejecutar_agente():
    """Ejecuta el agente de conciliación usando el loop agéntico de Claude."""
    print("\n" + "=" * 62)
    print("  AGENTE DE CONCILIACIÓN BANCARIA")
    print("  MI EMPRESA SA DE CV  ←→  BBVA México")
    print("  Periodo: 01 — 09 Mayo 2024")
    print("=" * 62 + "\n")

    mensaje_usuario = (
        f"Realiza la conciliación bancaria completa del periodo 01-09 Mayo 2024.\n\n"
        f"Archivos a conciliar:\n"
        f"  • Movimientos internos : {MOVIMIENTOS_FILE}\n"
        f"  • Extracto MT940       : {MT940_FILE}\n\n"
        f"Guarda el reporte en: {REPORTE_FILE}\n\n"
        f"Al finalizar proporciona un análisis ejecutivo que incluya:\n"
        f"  - Total de registros conciliados 1:1 y casos 1:N\n"
        f"  - Detalle del caso 1:N (pago combinado renta + CFE)\n"
        f"  - Verificación de que no hay registros no conciliados\n"
        f"  - Consistencia de saldos: el saldo inicial + movimientos netos = saldo final\n"
        f"  - Cualquier anomalía o hallazgo relevante"
    )

    messages = [{"role": "user", "content": mensaje_usuario}]
    iteracion = 0

    print("Iniciando agente...\n")

    while True:
        iteracion += 1
        print(f"[Iteración {iteracion}] Consultando Claude ({MODEL})...")

        response = CLIENT.messages.create(
            model=MODEL,
            max_tokens=8096,
            system=[
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                    "cache_control": {"type": "ephemeral"},  # cachear el system prompt
                }
            ],
            tools=TOOLS,
            messages=messages,
        )

        print(f"  Stop reason: {response.stop_reason} | "
              f"Input tokens: {response.usage.input_tokens} | "
              f"Output tokens: {response.usage.output_tokens}")

        tiene_tool_use = False
        tool_results = []

        for block in response.content:
            if block.type == "text" and block.text.strip():
                print(f"\n{'─' * 62}")
                print(block.text)
                print(f"{'─' * 62}\n")

            elif block.type == "tool_use":
                tiene_tool_use = True
                print(f"\n  → Herramienta: {block.name}")

                # Log compacto de los inputs
                for k, v in block.input.items():
                    if isinstance(v, list):
                        print(f"     {k}: [{len(v)} elementos]")
                    elif isinstance(v, dict):
                        if "resumen" in v:
                            r = v["resumen"]
                            print(f"     resultado: conciliados={r.get('conciliados_1_1', 0)}+{r.get('conciliados_1_n', 0)}, "
                                  f"nc_mov={r.get('no_conciliados_movimientos', 0)}, "
                                  f"nc_mt940={r.get('no_conciliados_mt940', 0)}")
                        else:
                            print(f"     {k}: {{...}}")
                    else:
                        print(f"     {k}: {v}")

                try:
                    resultado = ejecutar_herramienta(block.name, block.input)

                    # Log del resultado
                    if isinstance(resultado, list):
                        print(f"     ✓ {len(resultado)} registros parseados")
                    elif isinstance(resultado, dict) and "resumen" in resultado:
                        r = resultado["resumen"]
                        print(f"     ✓ Conciliados: {r['conciliados_1_1']} (1:1) + {r['conciliados_1_n']} (1:N) "
                              f"| No conciliados: {r['no_conciliados_movimientos']} mov / {r['no_conciliados_mt940']} MT940 "
                              f"| Tasa: {r['tasa_conciliacion']}%")
                    elif isinstance(resultado, str):
                        print(f"     ✓ Reporte generado — {len(resultado.splitlines())} líneas")
                    else:
                        print(f"     ✓ OK")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(resultado, ensure_ascii=False, default=str),
                    })

                except Exception as exc:
                    print(f"     ✗ ERROR: {exc}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "is_error": True,
                        "content": str(exc),
                    })

        # Condición de salida del loop
        if response.stop_reason == "end_turn" and not tiene_tool_use:
            print("\n✓ Agente completó el proceso.")
            break
        if not tiene_tool_use:
            print("\n✓ Sin más herramientas que ejecutar.")
            break

        # Continuar con los resultados
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    print(f"\n📄 Reporte guardado en: {REPORTE_FILE}\n")
    return str(REPORTE_FILE)


if __name__ == "__main__":
    ruta_reporte = ejecutar_agente()
    print(f"Proceso finalizado. Reporte disponible en:\n{ruta_reporte}")


NameError: name '__file__' is not defined

In [3]:
pip install anthropic


Note: you may need to restart the kernel to use updated packages.
